In [84]:
class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)


In [86]:
#Home Team

home_world_cup_starting_xi_vs_away = [
    ["Mike Maignan", "GK", "AC Milan", 88],
    ["Jules Kounde", "RB", "Barcelona", 84],
    ["William Saliba", "CB", "Arsenal", 88],
    ["Dayot Upamecano", "CB", "Bayern Munich", 87],
    ["Theo Hernandez", "LB", "Al Hilal", 82],
    ["Aurelien Tchouameni", "CDM", "Real Madrid", 86],
    ["Adrien Rabiot", "CDM", "AC Milan", 84],
    ["Ousmane Dembele", "RW", "Paris St-Germain", 89],
    ["Desire Doue", "CAM", "Paris St-Germain", 86],
    ["Kylian Mbappe", "LW", "Real Madrid", 91],
    ["Michael Olise", "RW", "Bayern Munich", 90],


] 
hometeam_world_cup_bench_vs_awayteam= [
    ["Bradley Barcola", "LW", "Paris St-Germain", 83],
    ["Rayan Cherki", "CAM", "Manchester City", 84],
]


In [88]:
gk = GoalkeeperProfile("Maignan", "France", "goalkeeper", 88)
gk.input_match_stats(
    minutes_played=90,
    saves=2,
    saves_inside_box=2,
    saves_outside_box=0,
    goals_conceded=1,
    xG_faced=0.17,
    goals_prevented=-0.83,
    total_passes=39,
    accurate_passes=28,
    total_long_balls=16,
    accurate_long_balls=6,
    touches=50,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Maignan's match rating: 6.80


In [22]:
player = PlayerProfile("Saliba", "France", "defender", 88)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=67,
    total_passes=72,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=4,
    interceptions=1,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Saliba's match rating: 7.10


In [24]:
player = PlayerProfile("Kounde", "France", "defender", 84)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=51,
    total_passes=59,
    expected_goals=0,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=1,
    total_long_balls=5,
    dispossessed=0,
    tackles_won=4,
    tackles_loss=0,
    clearances=4,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=5,
    duels_lost=5,
    ground_duels_won=4,
    ground_duels_total=5,
    aerial_duels_won=1,
    aerial_duels_total=5,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kounde's match rating: 7.01


In [26]:
player = PlayerProfile("Upamecano", "France", "defender", 87)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=70,
    total_passes=80,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=3,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=6,
    tackles_loss=0,
    clearances=1,
    interceptions=3,
    ball_recoveries=5,
    dribbled_past=0,
    duels_won=8,
    duels_lost=4,
    ground_duels_won=7,
    ground_duels_total=8,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Upamecano's match rating: 7.95


In [28]:
player = PlayerProfile("Hernandez", "France", "defender", 82)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=45,
    total_passes=48,
    expected_goals=0.12,
    expected_assists=0.11,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=2,
    tackles_won=2,
    tackles_loss=0,
    clearances=2,
    interceptions=1,
    ball_recoveries=5,
    dribbled_past=2,
    duels_won=4,
    duels_lost=5,
    ground_duels_won=2,
    ground_duels_total=7,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hernandez's match rating: 6.90


In [30]:
player = PlayerProfile("Tchouameni", "France", "midfielder", 86)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=62,
    total_passes=69,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=0,
    interceptions=2,
    ball_recoveries=6,
    dribbled_past=1,
    duels_won=6,
    duels_lost=1,
    ground_duels_won=3,
    ground_duels_total=4,
    aerial_duels_won=3,
    aerial_duels_total=3,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Tchouameni's match rating: 7.81


In [32]:
player = PlayerProfile("Rabiot", "France", "midfielder", 84)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=0,
    total_passes=0,
    expected_goals=0,
    expected_assists=0.33,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=5,
    total_long_balls=5,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=0,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Rabiot's match rating: 8.18


In [34]:
player = PlayerProfile("Olise", "France", "midfielder", 90)

player.input_match_stats(
    minutes=90,
    goals=0,
    big_chances_missed=1,
    assists=1,
    total_shots=2,
    shots_on_target=2,
    accurate_passes=51,
    total_passes=57,
    expected_goals=0.36,
    expected_assists=0.95,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=2,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=6,
    dribbled_past=1,
    duels_won=6,
    duels_lost=3,
    ground_duels_won=6,
    ground_duels_total=9,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Olise's match rating: 8.75


In [36]:
player = PlayerProfile("Doue", "France", "midfielder", 86)

player.input_match_stats(
    minutes=87,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=31,
    total_passes=36,
    expected_goals=0.08,
    expected_assists=0.16,
    successful_dribbles=1,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=2,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=4,
    duels_won=4,
    duels_lost=7,
    ground_duels_won=3,
    ground_duels_total=9,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Doue's match rating: 5.39


In [38]:
player = PlayerProfile("Dembele", "France", "forward", 89)

player.input_match_stats(
    minutes=80,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=26,
    total_passes=34,
    expected_goals=0.03,
    expected_assists=0.05,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=3,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=2,
    duels_lost=0,
    ground_duels_won=2,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Dembele's match rating: 5.95


In [42]:
player = PlayerProfile("Mbappe", "France", "forward", 91)

player.input_match_stats(
    minutes=90,
    goals=2,
    big_chances_missed=1,
    assists=0,
    total_shots=4,
    shots_on_target=4,
    accurate_passes=15,
    total_passes=16,
    expected_goals=0.76,
    expected_assists=0.02,
    successful_dribbles=1,
    total_dribbles=6,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=2,
    duels_lost=7,
    ground_duels_won=2,
    ground_duels_total=9,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Mbappe's match rating: 7.95


In [44]:
player = PlayerProfile("Barcola", "France", "midfielder", 83)

player.input_match_stats(
    minutes=10,
    goals=1,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=6,
    total_passes=7,
    expected_goals=0.43,
    expected_assists=0.06,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Barcola's match rating: 7.92


In [46]:
player = PlayerProfile("Cherki", "France", "midfielder", 84)

player.input_match_stats(
    minutes=3,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=8,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Cherki's match rating: 6.50


In [12]:
#AwayTeam
away_world_cup_starting_xi_vs_home = [
    ["Edouard Mendy", "GK", "Al-Ahli", 78],
    ["Krepin Diatta", "RB", "Monaco", 76],
    ["Moussa Niakhate", "CB", "Lyon", 79],
    ["Kalidou Koulibaly", "CB", "Chelsea", 82],
    ["El-Hadji Malick Diouf", "LB", "West Ham United", 79],
    ["Lamine Camara", "CDM", "Monaco", 80],
    ["Idrissa Gueye", "CDM", "Everton", 76],
    ["Pape Gueye", "CM", "Villarreal", 79],
    ["Ismaila Sarr", "RW", "Crystal Palace", 81],
    ["Sadio Mane", "LW", "Al-Nassr", 82],
    ["Nicolas Jackson", "ST", "Bayern Munich", 80],
]

awayteam_world_cup_bench_vs_hometeam = [
    ["Ibrahim Mbaye", "LW", "Paris St-Germain", 78],
    ["Iliman Ndiaye", "CAM", "Everton", 81],
    ["Habib Diarra", "CM", "Sunderland", 78],
    ["Pathe Ciss", "CDM", "Rayo Vallecano", 77],
    ["Bamba Dieng", "ST", "Lorient", 77]

]   

In [50]:
gk = GoalkeeperProfile("Mendy", "Senegal", "goalkeeper", 78)
gk.input_match_stats(
    minutes_played=90,
    saves=5,
    saves_inside_box=5,
    saves_outside_box=0,
    goals_conceded=3,
    xG_faced=2.06,
    goals_prevented=-0.94,
    total_passes=36,
    accurate_passes=31,
    total_long_balls=10,
    accurate_long_balls=5,
    touches=47,
    errors=1,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Mendy's match rating: 7.04


In [52]:
player = PlayerProfile("Diatta", "Senegal", "defender", 76)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=43,
    total_passes=50,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=3,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=2,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=5,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=9,
    duels_lost=1,
    ground_duels_won=8,
    ground_duels_total=8,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=3 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Diatta's match rating: 7.60


In [56]:
player = PlayerProfile("Koulibaly", "Senegal", "defender", 82)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=36,
    total_passes=44,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=5,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=3 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Koulibaly's match rating: 5.88


In [58]:
player = PlayerProfile("Niakhate", "Senegal", "defender", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=76,
    total_passes=83,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=3,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=3 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Niakhate's match rating: 6.47


In [60]:
player = PlayerProfile("Diouf", "Senegal", "defender", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=32,
    total_passes=43,
    expected_goals=0.03,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=3,
    accurate_long_balls=4,
    total_long_balls=7,
    dispossessed=2,
    tackles_won=2,
    tackles_loss=0,
    clearances=4,
    interceptions=1,
    ball_recoveries=6,
    dribbled_past=2,
    duels_won=6,
    duels_lost=5,
    ground_duels_won=2,
    ground_duels_total=7,
    aerial_duels_won=4,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=3 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Diouf's match rating: 6.47


In [62]:
player = PlayerProfile("P.Gueye", "Senegal", "midfielder", 79)

player.input_match_stats(
    minutes=83,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=42,
    total_passes=49,
    expected_goals=0,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=5,
    total_long_balls=8,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=2,
    interceptions=2,
    ball_recoveries=7,
    dribbled_past=0,
    duels_won=0,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

P.Gueye's match rating: 6.41


In [64]:
player = PlayerProfile("Camara", "Senegal", "midfielder", 80)

player.input_match_stats(
    minutes=76,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=26,
    total_passes=29,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=1,
    duels_lost=5,
    ground_duels_won=1,
    ground_duels_total=5,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Camara's match rating: 6.21


In [66]:
player = PlayerProfile("I.Gueye", "Senegal", "midfielder", 76)

player.input_match_stats(
    minutes=88,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=56,
    total_passes=61,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

I.Gueye's match rating: 7.30


In [68]:
player = PlayerProfile("Mane", "Senegal", "forward", 82)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=26,
    total_passes=30,
    expected_goals=0.03,
    expected_assists=0.02,
    successful_dribbles=1,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=2,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=4,
    duels_lost=6,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=1,
    aerial_duels_total=3,
    fouled=3,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Mane's match rating: 6.97


In [70]:
player = PlayerProfile("Sarr", "Senegal", "forward", 81)

player.input_match_stats(
    minutes=75,
    big_chances_missed=1,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=0,
    accurate_passes=11,
    total_passes=13,
    expected_goals=0.27,
    expected_assists=0.04,
    successful_dribbles=2,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sarr's match rating: 5.08


In [72]:
player = PlayerProfile("Jackson", "Senegal", "forard", 80)

player.input_match_stats(
    minutes=83,
    goals=0,
    big_chances_missed=1,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=12,
    total_passes=18,
    expected_goals=0.13,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=4,
    tackles_won=1,
    tackles_loss=0,
    clearances=2,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=2,
    duels_lost=12,
    ground_duels_won=1,
    ground_duels_total=9,
    aerial_duels_won=1,
    aerial_duels_total=5,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Jackson's match rating: 4.62


In [74]:
player = PlayerProfile("Mbaye", "Senegal", "forward", 78)

player.input_match_stats(
    minutes=15,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=7,
    total_passes=8,
    expected_goals=0.1,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=2,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Mbaye's match rating: 7.96


In [76]:
player = PlayerProfile("Ndiaye", "Senegal", "forward", 81)

player.input_match_stats(
    minutes=7,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=9,
    total_passes=10,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=2,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ndiaye's match rating: 7.30


In [78]:
player = PlayerProfile("Ciss", "midfielder", "Senegal", 77)

player.input_match_stats(
    minutes=2,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=12,
    total_passes=14,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ciss's match rating: 7.30


In [80]:
player = PlayerProfile("Diarra", "Senegal", "midfielder", 78)

player.input_match_stats(
    minutes=14,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=8,
    total_passes=9,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Diarra's match rating: 5.76


In [82]:
player = PlayerProfile("Dieng", "Senegal", "forward", 77)

player.input_match_stats(
    minutes=7,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=3,
    total_passes=3,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Dieng's match rating: 6.00
